# Ingest the whole corpus

`01_ingestion_colab.ipynb` walks one document through every stage so the pipeline is visible. This one just runs all of them, through `ingest_all.py`.

Two things worth knowing before starting.

**This takes hours, and Colab disconnects.** A free runtime idles out after roughly 90 minutes untouched, and caps around 12 hours regardless. 2,300 pages of parsing runs well past the first of those. The run is crash-safe — each document is committed as it finishes, and a re-run skips whatever already succeeded — so a disconnect costs you the document in flight, not the ones already done. Just re-run the same cell.

**Keep the tab open and visible.** Colab treats a background tab as idle. Leaving it in the foreground on a machine that does not sleep is the difference between one long run and five interrupted ones.

In [ ]:
# Where you've uploaded this project's files in your Drive — the rag/ package,
# requirements.txt, ingest_all.py, and pdfs/. Adjust if you put it elsewhere.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/05-rag'
%cd {PROJECT_DIR}
%pip install -q -r requirements.txt

In [ ]:
# Secrets from Colab's manager — key icon in the left sidebar. MUST run before
# any `from rag import ...`, since config.py reads every setting at import time.
import os
from google.colab import userdata

for name in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    os.environ[name] = userdata.get(name)

# The three settings validated across this corpus. Set here rather than read
# from Colab secrets — none of them is secret, and a plain assignment is one
# less thing to configure in two places.
os.environ["TABLE_CELL_MATCHING"] = "1"
os.environ["MIN_CHUNK_TOKENS"] = "150"
os.environ["FIX_HEADING_HIERARCHY"] = "1"

print("secrets loaded, ingestion settings applied")

## What would run

`--dry-run` parses nothing. It lists the PDFs found and what state each is in, which is the cheapest way to catch a wrong folder path before committing hours to a run.

In [ ]:
PDF_DIR = 'pdfs/clinical_trials_20'

!python ingest_all.py {PDF_DIR} --dry-run

## The run

Smallest document first — a systemic problem surfaces in the first few minutes rather than after the largest file has already burned an hour.

Already-succeeded documents are skipped automatically, on size and mtime, so re-running after a disconnect resumes rather than restarts. Nothing below needs changing between attempts.

**Cost, once, for the whole corpus:** roughly \$1–2 in embeddings at `text-embedding-3-small`, plus vision-model calls for figures. Figure descriptions are cached on the rendered image bytes, so a resumed run re-parses but does not re-pay for figures it has already described.

In [ ]:
!python ingest_all.py {PDF_DIR}

## What happened

`reports/_run.json` is the machine-readable record every resume decision is made from — one row per document, with its status, page count, record count and any error.

In [ ]:
import json
import pandas as pd

run = json.load(open('reports/_run.json'))
rows = run['rows'] if isinstance(run, dict) and 'rows' in run else run
df = pd.DataFrame(rows)

print(f"{len(df)} document(s)")
if 'status' in df.columns:
    print(df['status'].value_counts().to_string())
df

## Anything that failed

A failure is recorded, not raised — one bad document never stops the other nineteen. This is where they surface.

Re-running the ingest cell above retries exactly these; the successful documents are skipped.

In [ ]:
failed = df[df['status'] != 'ok'] if 'status' in df.columns else df.iloc[0:0]

if failed.empty:
    print("no failures")
else:
    for _, row in failed.iterrows():
        print(f"{row.get('file', '?')}")
        print(f"   {str(row.get('error', ''))[:300]}\n")

## Confirm what actually reached the index

The run log says what this process did. This asks Pinecone what is actually there — the two can disagree, and only the second one matters for retrieval.

In [ ]:
from rag import config
from rag import index as index_module

index = index_module.open_index()
stats = index.describe_index_stats()

print(f"index      : {config.INDEX_NAME}")
print(f"namespace  : {config.NAMESPACE!r}")
print(f"vectors    : {stats.total_vector_count}")